In [9]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Find project root automatically
cwd = Path.cwd().resolve()

ROOT = next(
    p for p in [cwd, *cwd.parents]
    if (p / "project1_newsvendor" / "src" / "data_synthetic.py").exists()
)

sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

from project1_newsvendor.src.data_synthetic import (
    load_synthetic,
    true_mean,
    true_quantile
)

from project1_newsvendor.src.newsvendor import (
    critical_ratio,
    newsvendor_cost
)

# Create figures directory
FIG = ROOT / "project1_newsvendor" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# Load data
df, truth = load_synthetic()
df = df.set_index("date")

Project root: C:\Users\bibek\UoS\Practice\Projects\newsvendor_predict-then-optimize


In [10]:
# ## 1. Basic hygiene: shape, missing values, contiguous dates
 
# %%
print("shape:", df.shape)
print("missing values:\n", df.isna().sum().to_string())
full_range = pd.date_range(df.index.min(), df.index.max(), freq="D")
print("all days present:", len(full_range) == len(df), "| duplicates:", df.index.duplicated().sum())
print("demand min / max:", df.demand.min(), df.demand.max())

shape: (1460, 6)
missing values:
 demand        0
dow           0
month         0
is_weekend    0
promo         0
holiday       0
all days present: True | duplicates: 0
demand min / max: 27 469


In [11]:
# ## 2. Is demand right-skewed?  (mean > median, long upper tail)
 
# %%
d = df["demand"]
print(f"mean {d.mean():.1f} | median {d.median():.1f} | q75 {d.quantile(.75):.1f} "
      f"| q90 {d.quantile(.90):.1f} | q95 {d.quantile(.95):.1f}")
print("skewness:", round(d.skew(), 2))

mean 67.7 | median 61.0 | q75 78.0 | q90 101.0 | q95 119.0
skewness: 3.6


In [12]:
# ## 3. Do the features matter?  (weekday, promo)  and is the spread feature-dependent?
# Heteroscedasticity check: the coefficient of variation (std/mean) should be higher
# on promo days and weekends, because we built sigma that way.
 
# %%
by_dow = df.groupby("dow")["demand"].agg(["mean", "median", "std"]).round(1)
by_promo = df.groupby("promo")["demand"].agg(["mean", "median", "std"]).round(1)
by_weekend = df.groupby("is_weekend")["demand"].agg(["mean", "median", "std"]).round(1)
for name, t in [("weekday (0=Mon)", by_dow), ("promo", by_promo), ("weekend", by_weekend)]:
    t["cv"] = (t["std"] / t["mean"]).round(2)
    print(f"\nBy {name}\n{t.to_string()}")


By weekday (0=Mon)
     mean  median   std    cv
dow                          
0    54.7    52.0  17.8  0.33
1    58.3    54.0  21.5  0.37
2    61.8    59.0  20.9  0.34
3    61.5    58.0  19.5  0.32
4    70.1    64.0  23.8  0.34
5    87.8    78.0  46.0  0.52
6    80.0    72.0  34.5  0.43

By promo
        mean  median   std    cv
promo                           
0       63.9    60.0  22.6  0.35
1      102.1    93.0  55.7  0.55

By weekend
            mean  median   std    cv
is_weekend                          
0           61.3    57.0  21.4  0.35
1           83.9    75.0  40.8  0.49


In [13]:
# ## 4. Autocorrelation: does yesterday carry information about today?
# Raw autocorrelation mixes in the weekly pattern, so check it after removing
# the weekday average. A positive value at lag 1 comes from the hidden level.
 
# %%
resid = np.log(d) - np.log(d).groupby(df["dow"]).transform("mean")
acf = {lag: round(float(resid.autocorr(lag)), 2) for lag in [1, 2, 7, 14]}
print("autocorrelation of weekday-adjusted log demand:", acf)

autocorrelation of weekday-adjusted log demand: {1: 0.39, 2: 0.33, 7: 0.28, 14: 0.25}


In [14]:
# ## 5. Benchmark costs (c_o = 1, c_u = 3, so CR = 0.75)
# These use the TRUE generating parameters, so they are not achievable by a fitted
# model, but they tell us the floor each method should approach.
#   A. order the global mean              (ignores features, ignores asymmetry)
#   B. order the global 75th percentile   (ignores features, respects asymmetry)
#   C. order the true conditional mean    (uses features, ignores asymmetry)
#   D. order the true conditional q75     (uses features AND asymmetry)  <- floor
 
# %%
c_o, c_u = 1.0, 3.0
cr = critical_ratio(c_o, c_u)
q_true = true_quantile(truth, cr)
 
rules = {
    "A. global mean": np.full(len(d), d.mean()),
    "B. global q75": np.full(len(d), d.quantile(cr)),
    "C. true conditional mean": true_mean(truth),
    "D. true conditional q75": q_true,
}
rows = []
for name, q in rules.items():
    cost = newsvendor_cost(q, d.to_numpy(), c_o, c_u)
    rows.append({"rule": name, "avg_cost": cost.mean()})
bench = pd.DataFrame(rows).set_index("rule")
bench["vs_A_pct"] = (100 * (bench["avg_cost"] / bench.loc["A. global mean", "avg_cost"] - 1)).round(1)
bench["avg_cost"] = bench["avg_cost"].round(2)
print(bench.to_string())
print(f"\ncoverage of true q75 (share of days demand <= order): {(d.to_numpy() <= q_true).mean():.3f}  (target {cr})")
bench.to_csv(ROOT / "project1_newsvendor" / "results" / "step2_benchmarks.csv")
 

                          avg_cost  vs_A_pct
rule                                        
A. global mean               40.37       0.0
B. global q75                37.83      -6.3
C. true conditional mean     27.94     -30.8
D. true conditional q75      24.22     -40.0

coverage of true q75 (share of days demand <= order): 0.754  (target 0.75)


In [15]:
# ## 6. Figures
 
# %%
fig, ax = plt.subplots(figsize=(10, 3.5))
d.iloc[:365].plot(ax=ax, lw=1)
promo_days = df.index[df["promo"] == 1]
ax.scatter(promo_days[promo_days < d.index[365]], d.loc[promo_days[promo_days < d.index[365]]],
           color="red", s=12, label="promo day", zorder=3)
ax.set_title("Daily demand, first year"); ax.set_ylabel("demand"); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "02_demand_first_year.png", dpi=150); plt.close(fig)
 
fig, ax = plt.subplots(figsize=(6, 4))
df.boxplot(column="demand", by="dow", ax=ax)
ax.set_title("Demand by weekday (0 = Monday)"); fig.suptitle("")
fig.tight_layout(); fig.savefig(FIG / "02_demand_by_weekday.png", dpi=150); plt.close(fig)
 
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(d, bins=60, color="lightsteelblue")
for val, lab, col in [(d.mean(), "mean", "green"), (d.median(), "median", "orange"),
                      (d.quantile(cr), "q75", "red")]:
    ax.axvline(val, color=col, ls="--", label=f"{lab} {val:.1f}")
ax.set_title("Demand distribution (right-skewed)"); ax.set_xlabel("demand"); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "02_demand_histogram.png", dpi=150); plt.close(fig)
 
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0, 250, 60)
ax.hist(df.loc[df.promo == 0, "demand"], bins=bins, alpha=.6, density=True, label="no promo")
ax.hist(df.loc[df.promo == 1, "demand"], bins=bins, alpha=.6, density=True, label="promo")
ax.set_title("Promo shifts AND widens the distribution"); ax.set_xlabel("demand"); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "02_promo_vs_no_promo.png", dpi=150); plt.close(fig)
 
print("figures saved to", FIG)

figures saved to C:\Users\bibek\UoS\Practice\Projects\newsvendor_predict-then-optimize\project1_newsvendor\figures
